<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Required Libraries
!pip install -q langchain langchain-community chromadb pypdf sentence-transformers langchain-ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [2]:
# Install and Run Ollama in Background

# Install required system dependency (zstd) for Ollama extraction
!apt-get update -qq && apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
# Install Ollama locally in the Colab instance
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# Start the Ollama server in the background using nohup
import subprocess
import time

# Starting Ollama server in the background
subprocess.Popen(["nohup", "ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to start
time.sleep(5)

In [5]:
# Pull the specific embedding model we need for the pipeline
# Pulling the nomic-embed-text model
!ollama pull nomic-embed-text
# Ollama setup is complete

In [6]:
# Data Ingestion Pipeline (The Core Logic)
import os
import shutil
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

In [7]:
# --- CONFIGURATION ---

# The folder where our uploaded methology PDF files in Colab
PDF_DOCS_DIR = "methodology_pdfs/"

# The folder where ChromaDB will save the vectorized database
CHROMA_DB_DIR = "vector_db/methodology_db"

# The embedding model we just pulled via Ollama
EMBEDDING_MODEL_NAME = "nomic-embed-text"

In [8]:
def prepare_vector_database():
    """
    Reads PDFs, chunks them, and creates a local Vector Database (ChromaDB).
    This script prepares the 'Methodology RAG' component for Node 1.
    """
    print(f"--- Phase 0: Data Ingestion Started ---")

    # 1. Clear previous database if it exists (for a clean run)
    if os.path.exists(CHROMA_DB_DIR):
        print(f"[*] Deleting old database at: {CHROMA_DB_DIR}")
        shutil.rmtree(CHROMA_DB_DIR)

    # 2. Check if the PDF folder exists and has files
    if not os.path.exists(PDF_DOCS_DIR) or len(os.listdir(PDF_DOCS_DIR)) == 0:
        print(f"[!] ERROR: Folder '{PDF_DOCS_DIR}' not found or is empty.")
        print(f"[!] Please create the folder in Colab and upload your methodology PDFs")
        return

    # 3. Load the PDFs using LangChain
    print(f"[*] Loading PDF documents from: {PDF_DOCS_DIR}")
    loader = PyPDFDirectoryLoader(PDF_DOCS_DIR)
    documents = loader.load()

    if not documents:
        print("[!] No PDF documents could be read. Exiting.")
        return

    print(f"[*] Successfully loaded {len(documents)} pages from the PDFs.")

    # 4. Chunking (Text Splitting)
    # Splitting large PDFs into smaller, meaningful chunks (approx 800 characters)
    # Overlap prevents sentences from being cut abruptly.
    print("[*] Splitting documents into chunks for the Vector DB...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"[*] Split the PDFs into {len(chunks)} contextual chunks.")

    # 5. Initialize the Embedding Model
    print(f"[*] Initializing Ollama Embedding Model: '{EMBEDDING_MODEL_NAME}'")
    try:
        embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
    except Exception as e:
        print(f"[!] Error connecting to Ollama. Error: {e}")
        return

    # 6. Create and Save the Vector Database (ChromaDB)
    print(f"[*] Generating embeddings and saving to ChromaDB at: {CHROMA_DB_DIR}")
    print(f"[*] (Please wait, this might take 1-2 minutes depending on PDF sizes...)")

    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DB_DIR
    )

    # Force save the database to the Colab disk
    vector_db.persist()
    print(f"--- Data Ingestion Complete! ---")
    print(f"[*] The vector database is now saved in the '{CHROMA_DB_DIR}' folder.")

In [9]:
# Run the pipeline
prepare_vector_database()

--- Phase 0: Data Ingestion Started ---
[*] Loading PDF documents from: methodology_pdfs/
[*] Successfully loaded 377 pages from the PDFs.
[*] Splitting documents into chunks for the Vector DB...
[*] Split the PDFs into 1143 contextual chunks.
[*] Initializing Ollama Embedding Model: 'nomic-embed-text'
[*] Generating embeddings and saving to ChromaDB at: vector_db/methodology_db
[*] (Please wait, this might take 1-2 minutes depending on PDF sizes...)
--- Data Ingestion Complete! ---
[*] The vector database is now saved in the 'vector_db/methodology_db' folder.


/tmp/ipykernel_1891/2481239435.py:61: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [11]:
# Zip the Database for GitHub Download
import shutil
import os

print("[*] Zipping the vector database for download...")
if os.path.exists("vector_db"):
    shutil.make_archive("vector_db", 'zip', "vector_db")
    print("[*] Success!")
else:
    print("[!] Error: vector_db folder not found.")

[*] Zipping the vector database for download...
[*] Success!
